# 🧑‍⚖️ Human-in-the-Loop Patterns in LangGraph

## Learning Objectives
In this notebook, you will learn:
1. **Interrupting a graph** - use `interrupt_before` to pause execution right before a chosen node runs.
2. **Inspecting paused state** - use `app.get_state()` to see what a graph is waiting on and which node runs next.
3. **Injecting human input** - use `app.update_state()` to write human decisions (approve/reject/feedback) into the graph's state.
4. **Resuming execution** - call `app.invoke(None, config)` to continue a paused graph from its last checkpoint.
5. **Building review loops** - combine interrupts with conditional edges to model multi-round human review cycles.

## Prerequisites
- Familiarity with LangGraph's `StateGraph`, nodes, and conditional edges (see `01`-`05` in this folder).
- An `OPENAI_API_KEY` set in a `.env` file at the project root.
- `langgraph`, `langchain-openai`, and `python-dotenv` installed.

> Converted from `06_human_in_loop.py` - part of **03 LangGraph Fundamentals**.

---
## 🔧 Setup: Imports, Environment, and Console Helpers

This section imports LangGraph's `StateGraph` machinery and `MemorySaver` checkpointer,
initializes a shared LLM, and loads environment variables from `.env`. It also defines
two small console-printing helpers used throughout the demos to make each interrupt/resume
step visually easy to follow in the output.

In [1]:
# ============================================================================
# ENVIRONMENT SETUP: Imports, .env Loading, and LLM Initialization
# ============================================================================
from typing import Literal

from typing_extensions import TypedDict

from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import END, START, StateGraph

load_dotenv()

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
print(f"✅ LLM initialized: {llm.model_name}")

✅ LLM initialized: gpt-4o-mini


### 🖨️ Console Helpers: `phase_banner` and `step_print`

Two tiny formatting helpers used purely for readability in the demo output below —
`phase_banner` prints a boxed header for each numbered phase of a workflow, and
`step_print` prints a single labeled step with an icon. Neither affects graph logic.

In [2]:
# ============================================================================
# CONSOLE HELPERS: phase_banner and step_print
# ============================================================================
def phase_banner(phase_num: int, title: str):
    """Print a boxed header marking a new phase of a demo workflow."""
    print(f"\n{'=' * 55}")
    print(f"  PHASE {phase_num}: {title}")
    print(f"{'=' * 55}")


def step_print(icon: str, label: str, detail: str = ""):
    """Print a single labeled step with an icon prefix."""
    print(f"\n{icon} [{label}] {detail}")

---
## ⏸️ Part 1: Interrupt-and-Approve Workflow

This part builds the simplest human-in-the-loop pattern: run a graph until a chosen
node, freeze it there with `interrupt_before`, let a human inspect the in-flight state
and inject a decision, then resume. The workflow drafts a message, pauses for approval,
and revises the draft if the human requests changes.

### Key Concepts:
- **`interrupt_before=[...]`**: a list of node names the compiled graph pauses in front of.
- **`app.get_state(config)`**: reads the frozen state and the next node(s) to run.
- **`app.update_state(config, {...})`**: merges human-provided values into the state before resuming.
- **`app.invoke(None, config)`**: resuming with `None` as input continues from the last checkpoint instead of starting over.

### 🗂️ State Schema: `ApprovalState`

The state carried through the approval graph: the original `request`, the LLM-generated
`draft`, the human's `approved` decision, any `feedback` text, and the `final` output.

In [3]:
# ============================================================================
# STATE SCHEMA: ApprovalState
# ============================================================================
class ApprovalState(TypedDict):
    request: str
    draft: str
    approved: bool
    feedback: str
    final: str

### ⚙️ Demo Function: `demo_interrupt_for_approval`

Builds a `draft -> approval -> finalize` graph, compiles it with `interrupt_before=["approval"]`,
then walks through three phases: running until the interrupt, inspecting the paused state, and
injecting human feedback before resuming to completion.

In [4]:
# ============================================================================
# DEMO FUNCTION: demo_interrupt_for_approval
# ============================================================================
def demo_interrupt_for_approval():
    """Interrupt execution for human approval."""

    def create_draft(state: ApprovalState) -> dict:
        step_print("📝", "DRAFT NODE", "Entering create_draft node...")
        print(f"   Request: \"{state['request']}\"")
        print(f"   Calling LLM to generate draft...")
        response = llm.invoke(f"Create a professional response for: {state['request']}")
        print(f"   Draft generated ({len(response.content.split())} words)")
        print(f"   Preview: {response.content[:100]}...")
        return {"draft": response.content}

    def wait_for_approval(state: ApprovalState) -> dict:
        step_print("👁️", "APPROVAL NODE", "Entering wait_for_approval node...")
        print(f"   Approved: {state['approved']}")
        print(
            f"   Feedback: '{state['feedback']}'"
            if state["feedback"]
            else "   Feedback: (none yet)"
        )
        # This node is where we'll interrupt
        return state

    def finalize(state: ApprovalState) -> dict:
        step_print("📦", "FINALIZE NODE", "Entering finalize node...")
        print(f"   Approved: {state['approved']}")
        if state["approved"]:
            print(f"   Action: Using draft as-is (human approved)")
            return {"final": state["draft"]}
        else:
            print(f"   Action: Revising draft based on feedback...")
            print(f"   Feedback: \"{state['feedback']}\"")
            # Incorporate feedback
            response = llm.invoke(
                f"Revise this draft based on feedback:\n\n"
                f"Draft: {state['draft']}\n\n"
                f"Feedback: {state['feedback']}"
            )
            print(f"   Revised draft generated ({len(response.content.split())} words)")
            return {"final": response.content}

    graph = StateGraph(ApprovalState)
    graph.add_node("draft", create_draft)
    graph.add_node("approval", wait_for_approval)
    graph.add_node("finalize", finalize)
    graph.add_edge(START, "draft")
    graph.add_edge("draft", "approval")
    graph.add_edge("approval", "finalize")
    graph.add_edge("finalize", END)

    # Compile with checkpointer and interrupt
    memory = MemorySaver()
    app = graph.compile(
        checkpointer=memory, interrupt_before=["approval"]  # Pause before this node
    )

    print("\n" + "=" * 55)
    print("  HUMAN-IN-THE-LOOP: APPROVAL WORKFLOW")
    print("=" * 55)
    print("\n   Graph: START -> [draft] -> ⏸️ -> [approval] -> [finalize] -> END")
    print("   Interrupt set BEFORE: 'approval' node")

    # Configuration for this thread
    config = {"configurable": {"thread_id": "demo-1"}}

    # ─── PHASE 1: Run until interrupt ───
    phase_banner(1, "RUN UNTIL INTERRUPT")
    print("   Calling app.invoke() with initial state...")
    print("   The graph will run until it hits the interrupt point.\n")

    result = app.invoke(
        {
            "request": "Write a thank-you email for a job interview",
            "draft": "",
            "approved": False,
            "feedback": "",
            "final": "",
        },
        config,
    )

    step_print("⏸️", "PAUSED", "Graph execution interrupted!")
    print(f"   Draft is ready: {result['draft'][:150]}...")
    print(f"   Final is empty: '{result['final']}'")
    print(f"\n   The graph is now FROZEN. Waiting for human input.")
    print(f"   In a real app, your frontend would show the draft here.")

    # ─── PHASE 2: Inspect paused state ───
    phase_banner(2, "INSPECT PAUSED STATE")
    current_state = app.get_state(config)
    print(f"   app.get_state(config) tells us:")
    print(f"   Next node(s): {current_state.next}")
    print(f"   State keys: {list(current_state.values.keys())}")
    print(f"   Draft filled: {'Yes' if current_state.values['draft'] else 'No'}")
    print(f"   Approved: {current_state.values['approved']}")
    print(f"   Final filled: {'Yes' if current_state.values['final'] else 'No'}")

    # ─── PHASE 3: Human provides feedback and resume ───
    phase_banner(3, "HUMAN INJECTS FEEDBACK + RESUME")
    feedback_text = (
        "Make it more concise and add specific mention of the company culture"
    )
    print(f"   Human decision: REJECT (request changes)")
    print(f'   Human feedback: "{feedback_text}"')
    print(f"\n   Calling app.update_state() to inject human input...")

    # Update state with human input
    app.update_state(
        config, {"approved": False, "feedback": feedback_text}  # Request changes
    )
    print(f"   State updated. approved=False, feedback set.")
    print(f"\n   Calling app.invoke(None, config) to RESUME...")
    print(f"   (None means 'no new input, just continue from checkpoint')\n")

    # Continue execution
    final_result = app.invoke(None, config)

    # ─── RESULT ───
    step_print("✅", "WORKFLOW COMPLETE", "")
    print(f"   Final result ({len(final_result['final'].split())} words):")
    print(f"   {final_result['final'][:200]}...")
    print(f"\n   Graph path taken:")
    print(
        f"   START -> [draft] -> ⏸️ PAUSE -> human feedback -> [approval] -> [finalize] -> END"
    )

---
## 🔁 Part 2: Iterative Review Workflow

This part extends the pattern into a loop: a document is submitted for review, a
router checks whether it was approved, and if not it loops back through revision and
re-submission — pausing before every submission — until the reviewer finally approves it.

### Key Insight
Because `interrupt_before=["submit"]` fires on *every* pass through the `submit` node,
the same interrupt point naturally supports an arbitrary number of review rounds without
any extra bookkeeping in the graph definition itself.

### 🗂️ State Schema: `ReviewState`

Tracks the `document` under review, the running list of `review_comments`, how many
`revision_count` passes have happened, and the current `status` (drives routing).

In [5]:
# ============================================================================
# STATE SCHEMA: ReviewState
# ============================================================================
class ReviewState(TypedDict):
    document: str
    review_comments: list[str]
    revision_count: int
    status: str

### ⚙️ Demo Function: `demo_iterative_review`

Builds a `submit -> [router] -> apply -> submit (loop) ... -> done` graph, compiled with
`interrupt_before=["submit"]`, then drives it through four rounds of human review: an
initial pause, two rounds of requested revisions, and a final approval.

In [6]:
# ============================================================================
# DEMO FUNCTION: demo_iterative_review
# ============================================================================
def demo_iterative_review():
    """Multiple rounds of human review."""

    def submit_for_review(state: ReviewState) -> dict:
        step_print("📋", "SUBMIT NODE", f"Round {state['revision_count'] + 1}")
        print(f"   Status incoming: '{state['status']}'")
        print(f"   Setting status to 'pending_review'")
        print(f"   Document preview: {state['document'][:100]}...")
        return {"status": "pending_review"}

    def apply_feedback(state: ReviewState) -> dict:
        step_print(
            "🔧", "APPLY FEEDBACK NODE", f"Revision #{state['revision_count'] + 1}"
        )
        if not state["review_comments"]:
            print(f"   No comments to apply. Passing through.")
            return state
        feedback = state["review_comments"][-1]
        print(f'   Feedback to apply: "{feedback}"')
        print(f"   Current document: {state['document'][:80]}...")
        print(f"   Calling LLM to revise...")
        response = llm.invoke(
            f"Revise this document based on feedback:\n\n"
            f"Document: {state['document']}\n\n"
            f"Feedback: {feedback}"
        )
        print(f"   Revised document ({len(response.content.split())} words)")
        print(f"   Preview: {response.content[:100]}...")
        return {
            "document": response.content,
            "revision_count": state["revision_count"] + 1,
            "status": "revised",
        }

    def route_after_review(state: ReviewState) -> Literal["apply", "done"]:
        step_print("🔀", "ROUTER", f"Checking status: '{state['status']}'")
        if state["status"] == "approved":
            print(f"   Decision: APPROVED -> routing to 'done' node")
            return "done"
        print(f"   Decision: NOT APPROVED -> routing to 'apply' node")
        return "apply"

    def finalize(state: ReviewState) -> dict:
        step_print("🏁", "DONE NODE", "Finalizing document")
        print(f"   Total revisions: {state['revision_count']}")
        print(f"   Final document: {state['document'][:100]}...")
        return {"status": "finalized"}

    graph = StateGraph(ReviewState)
    graph.add_node("submit", submit_for_review)
    graph.add_node("apply", apply_feedback)
    graph.add_node("done", finalize)
    graph.add_edge(START, "submit")
    graph.add_conditional_edges(
        "submit", route_after_review, {"apply": "apply", "done": "done"}
    )
    graph.add_edge("apply", "submit")  # Loop for more reviews
    graph.add_edge("done", END)

    memory = MemorySaver()
    app = graph.compile(checkpointer=memory, interrupt_before=["submit"])

    print("\n" + "=" * 55)
    print("  HUMAN-IN-THE-LOOP: ITERATIVE REVIEW WORKFLOW")
    print("=" * 55)
    print("\n   Graph: START -> ⏸️ -> [submit] -> [ROUTER]")
    print("                                       ├── approved -> [done] -> END")
    print(
        "                                       └── else -> [apply] -> ⏸️ [submit] (LOOP)"
    )
    print("   Interrupt set BEFORE: 'submit' node (fires EVERY loop)")

    config = {"configurable": {"thread_id": "review-1"}}

    # ─── ROUND 0: Initial submission ───
    phase_banner(0, "INITIAL SUBMISSION")
    print("   Sending initial document into the graph...")
    print('   Document: "AI is technology that helps computers think."\n')

    result = app.invoke(
        {
            "document": "AI is technology that helps computers think.",
            "review_comments": [],
            "revision_count": 0,
            "status": "",
        },
        config,
    )

    step_print("⏸️", "PAUSED", "Graph hit interrupt_before='submit'")
    print(f"   Document ready for review: \"{result['document']}\"")
    print(f"   Revisions so far: {result['revision_count']}")
    current_state = app.get_state(config)
    print(f"   Next node: {current_state.next}")
    print(f"\n   Waiting for human reviewer...")

    # ─── ROUND 1: Reviewer wants changes ───
    phase_banner(1, "REVIEWER REQUESTS CHANGES")
    feedback_1 = "Add more technical depth and examples"
    print(f'   Reviewer says: "{feedback_1}"')
    print(f"   Reviewer sets status: 'needs_revision'")
    print(f"\n   Calling app.update_state() to inject review...")

    app.update_state(
        config, {"review_comments": [feedback_1], "status": "needs_revision"}
    )
    print(f"   State updated. Calling app.invoke(None) to resume...\n")
    result = app.invoke(None, config)

    step_print("⏸️", "PAUSED AGAIN", "Graph looped back to 'submit' and paused")
    print(f"   Revised document: {result['document'][:150]}...")
    print(f"   Revisions so far: {result['revision_count']}")
    current_state = app.get_state(config)
    print(f"   Next node: {current_state.next}")
    print(f"\n   Waiting for human reviewer again...")

    # ─── ROUND 2: Reviewer wants more changes ───
    phase_banner(2, "REVIEWER REQUESTS MORE CHANGES")
    feedback_2 = "Good improvement! Now add a concrete example of neural networks"
    print(f'   Reviewer says: "{feedback_2}"')
    print(f"   Reviewer sets status: 'needs_revision'")
    app.update_state(
        config, {"review_comments": [feedback_2], "status": "needs_revision"}
    )
    print(f"   Resuming graph...\n")
    result = app.invoke(None, config)

    step_print("⏸️", "PAUSED AGAIN", "Graph looped back to 'submit' and paused")
    print(f"   Revised document: {result['document'][:150]}...")
    print(f"   Revisions so far: {result['revision_count']}")

    # ─── ROUND 3: Reviewer approves ───
    phase_banner(3, "REVIEWER APPROVES")
    print(f'   Reviewer says: "Looks great!"')
    print(f"   Reviewer sets status: 'approved'")
    app.update_state(config, {"status": "approved"})
    print(f"   Resuming graph for final time...\n")
    final = app.invoke(None, config)

    # ─── FINAL SUMMARY ───
    step_print("✅", "WORKFLOW COMPLETE", "")
    print(f"   Final status: {final['status']}")
    print(f"   Total revisions: {final['revision_count']}")
    print(f"   Final document: {final['document'][:200]}...")
    print(f"\n   Full timeline:")
    print(f"   Round 0: START -> ⏸️ (human reviews initial doc)")
    print(f"   Round 1: resume -> [submit] -> [apply] -> ⏸️ (human reviews revision 1)")
    print(f"   Round 2: resume -> [submit] -> [apply] -> ⏸️ (human reviews revision 2)")
    print(f"   Round 3: resume -> [submit] -> [done] -> END (human approved!)")

---
## ▶️ Run the Demos

The original `__main__` guard, kept verbatim. Jupyter sets `__name__` to `"__main__"`, so
this cell runs both demos as-is when executed.

In [7]:
# ============================================================================
# RUN: Execute Both Demos
# ============================================================================
if __name__ == "__main__":
    print("=" * 55)
    print("  Demo 1: Interrupt for Approval")
    print("=" * 55)
    demo_interrupt_for_approval()

    print("\n" + "=" * 55)
    print("  Demo 2: Iterative Review")
    print("=" * 55)
    demo_iterative_review()

  Demo 1: Interrupt for Approval

  HUMAN-IN-THE-LOOP: APPROVAL WORKFLOW

   Graph: START -> [draft] -> ⏸️ -> [approval] -> [finalize] -> END
   Interrupt set BEFORE: 'approval' node

  PHASE 1: RUN UNTIL INTERRUPT
   Calling app.invoke() with initial state...
   The graph will run until it hits the interrupt point.


📝 [DRAFT NODE] Entering create_draft node...
   Request: "Write a thank-you email for a job interview"
   Calling LLM to generate draft...
   Draft generated (185 words)
   Preview: Subject: Thank You for the Opportunity

Dear [Interviewer's Name],

I hope this message finds you we...

⏸️ [PAUSED] Graph execution interrupted!
   Draft is ready: Subject: Thank You for the Opportunity

Dear [Interviewer's Name],

I hope this message finds you well. I wanted to take a moment to express my sincer...
   Final is empty: ''

   The graph is now FROZEN. Waiting for human input.
   In a real app, your frontend would show the draft here.

  PHASE 2: INSPECT PAUSED STATE
   app.get_

---
## 📝 Summary

In this notebook, we learned:

### 1. Interrupt-and-Approve Pattern
- **`interrupt_before=["node_name"]`**: freezes graph execution immediately before the named node runs.
- **`app.get_state(config)`**: inspects a paused graph's current values and the next node(s) queued to run.
- **`app.update_state(config, {...})`**: injects human decisions (approval, feedback) into the frozen state.
- **`app.invoke(None, config)`**: resumes execution from the checkpoint without re-running earlier nodes.

### 2. Iterative Review Loops
- A conditional edge (`route_after_review`) combined with an interrupt on the loop's entry node (`submit`)
  lets the same pause point serve an unbounded number of human review rounds.
- Each `update_state` + `invoke(None, config)` pair represents one full round-trip between the graph and a human reviewer.
- `MemorySaver` checkpointing is what makes pausing and resuming possible — state persists across calls keyed by `thread_id`.

### Next Steps
- Explore how these patterns extend to production settings with persistent checkpointers (e.g., Postgres) instead of `MemorySaver`.
- See the next notebooks in `03_LangGraph_Fundamentals/02_Core_Capabilities/` for streaming, subgraphs, and retry patterns that often combine with human-in-the-loop review.